# SIGNAL//SILENCE — Hidden Signal Discovery Lab
### Engineering the unseen with machine learning, anomaly detection, temporal drift analysis, and uncertainty exploration

**Research question:** Can a combined analytical workflow discover subtle anomalies, emerging changes, and uncertain predictions that ordinary summary statistics may overlook?

> This self-contained project generates synthetic sensor data. Results demonstrate methodology only and are not real-world operational findings.

## 1. Project architecture
1. Generate a time-dependent dataset with normal patterns, gradual drift, a short shock, and injected anomalies.
2. Audit and visualize the data.
3. Detect anomalies without labels using Isolation Forest.
4. Predict a synthetic event using a temporal train/test split.
5. Explore calibration, model disagreement, feature importance, and weak-signal ranking.
6. Export results and document limitations.

In [ ]:
import warnings, sys, subprocess, importlib.util
warnings.filterwarnings("ignore")
needed=["numpy","pandas","matplotlib","seaborn","scikit-learn"]
missing=[x for x in needed if importlib.util.find_spec(x) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install",*missing])
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score,
 confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, precision_recall_curve,
 f1_score, precision_score, recall_score)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=27
rng=np.random.default_rng(SEED)
sns.set_theme(style="whitegrid",context="notebook")
plt.rcParams["figure.figsize"]=(11,5)
print("Environment ready.")

## 2. Generate a synthetic time-dependent system
We simulate six sensor signals with periodic behaviour, gradual drift, a short disturbance, and rare injected point anomalies.

In [ ]:
N=9000
t=np.arange(N); phase=t/N
daily=np.sin(2*np.pi*t/240); weekly=np.sin(2*np.pi*t/1100); slow=np.sin(2*np.pi*t/2100)
drift=np.clip((phase-.55)/.25,0,1)
shock=((phase>.78)&(phase<.805)).astype(float)
load=50+7*daily+4*weekly+12*drift+15*shock+rng.normal(0,2.2,N)
temperature=28+2.5*slow+.8*drift+2*shock+rng.normal(0,.7,N)
latency=110+.55*load+5*temperature+16*drift+45*shock+rng.normal(0,9,N)
vibration=1.2+.012*load+.18*shock+.08*drift+rng.normal(0,.08,N)
throughput=900-3.2*load-1.4*latency+25*weekly-85*shock+rng.normal(0,24,N)
packet_loss=np.clip(.15+.008*np.maximum(latency-150,0)+.04*shock+rng.normal(0,.07,N),0,1.5)
injected=np.zeros(N,dtype=int)
idx=rng.choice(np.arange(100,N-100),115,replace=False); injected[idx]=1
load[idx]+=rng.choice([-1,1],len(idx))*rng.uniform(18,35,len(idx))
temperature[idx]+=rng.choice([-1,1],len(idx))*rng.uniform(3,6,len(idx))
vibration[idx]+=rng.uniform(.4,1,len(idx))
logit=-3+.035*(load-55)+.012*(latency-145)+1.6*(vibration-1.8)+2.1*packet_loss+1.4*shock+.9*drift
event=rng.binomial(1,1/(1+np.exp(-logit)))
df=pd.DataFrame({"time_index":t,"load":load,"temperature":temperature,"latency":latency,
 "vibration":vibration,"throughput":throughput,"packet_loss":packet_loss,
 "gradual_drift":drift,"shock_window":shock,"injected_anomaly":injected,"event":event})
for col,frac in [("temperature",.018),("latency",.012),("throughput",.02)]:
    miss=rng.choice(df.index,int(N*frac),replace=False); df.loc[miss,col]=np.nan
print("Shape:",df.shape,"| event rate:",round(df.event.mean(),3))
display(df.head())

## 3. Data quality audit

In [ ]:
audit=pd.DataFrame({"dtype":df.dtypes.astype(str),"missing":df.isna().sum(),
 "missing_pct":(100*df.isna().mean()).round(2),"unique":df.nunique()})
display(audit)
print("Duplicate rows:",df.duplicated().sum(),"| Injected anomalies:",df.injected_anomaly.sum())
fig,ax=plt.subplots(1,2,figsize=(13,4))
df.isna().mean().sort_values(ascending=False).plot(kind="bar",ax=ax[0],title="Missingness rate")
sns.countplot(data=df,x="event",ax=ax[1]); ax[1].set_title("Synthetic event balance")
plt.tight_layout(); plt.show()

## 4. Temporal signal exploration
The shaded intervals indicate gradual drift and the short shock window.

In [ ]:
fig,axs=plt.subplots(4,1,figsize=(14,11),sharex=True)
for ax,col in zip(axs,["load","latency","vibration","throughput"]):
    ax.plot(t,df[col],lw=.65)
    ax.axvspan(int(.55*N),int(.80*N),alpha=.10,color="orange")
    ax.axvspan(int(.78*N),int(.805*N),alpha=.18,color="red")
    ax.set_ylabel(col)
axs[-1].set_xlabel("Time index")
fig.suptitle("Signals over time: baseline, drift, and disturbance")
plt.tight_layout(); plt.show()

## 5. Correlation structure

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(df.drop(columns="time_index").corr(numeric_only=True),cmap="vlag",center=0,annot=True,fmt=".2f")
plt.title("Correlation map"); plt.tight_layout(); plt.show()

## 6. Rolling statistics reveal weak signals

In [ ]:
window=180
roll=df[["load","latency","vibration","throughput"]].rolling(window,min_periods=window//2)
means=roll.mean(); stds=roll.std()
fig,axs=plt.subplots(2,2,figsize=(14,8),sharex=True)
for ax,col in zip(axs.ravel(),means.columns):
    ax.plot(t,df[col],alpha=.2,lw=.6,label="raw")
    ax.plot(t,means[col],lw=1.5,label="rolling mean")
    ax.fill_between(t,means[col]-stds[col],means[col]+stds[col],alpha=.15)
    ax.set_title(f"{col}: rolling mean ± 1 SD")
axs[0,0].legend(); plt.tight_layout(); plt.show()

## 7. Unsupervised anomaly detection
Isolation Forest learns from the first half of the timeline, treated as a baseline, then scores all observations. Legitimate later drift may also appear unusual.

In [ ]:
features=["load","temperature","latency","vibration","throughput","packet_loss"]
baseline_end=int(.50*N)
imputer=SimpleImputer(strategy="median")
scaler=StandardScaler()
X_all=imputer.fit_transform(df[features])
X_base=scaler.fit_transform(X_all[:baseline_end])
X_scaled=scaler.transform(X_all)
iso=IsolationForest(n_estimators=300,contamination=.025,random_state=SEED,n_jobs=-1)
iso.fit(X_base)
df["anomaly_score"]=-iso.decision_function(X_scaled)
df["anomaly_flag"]=(iso.predict(X_scaled)==-1).astype(int)
plt.figure(figsize=(14,4))
plt.plot(t,df.anomaly_score,lw=.7)
plt.axhline(np.quantile(df.anomaly_score[:baseline_end],.975),ls="--",label="baseline 97.5th percentile")
plt.title("Anomaly score over time"); plt.xlabel("Time"); plt.legend(); plt.show()

## 8. Evaluate anomaly flags against injected anomalies
This controlled comparison is possible because the synthetic anomaly locations are known.

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(df.injected_anomaly,df.anomaly_flag,zero_division=0))
print("Anomaly ROC-AUC:",round(roc_auc_score(df.injected_anomaly,df.anomaly_score),3))
ConfusionMatrixDisplay.from_predictions(df.injected_anomaly,df.anomaly_flag,cmap="Blues")
plt.title("Injected anomaly detection"); plt.show()

## 9. Supervised event prediction with a temporal holdout
The first 75% of observations train the model; the final 25% test it. Preprocessing is fitted only on the training window.

In [ ]:
split=int(.75*N)
X_train=df.loc[:split-1,features]; y_train=df.loc[:split-1,"event"]
X_test=df.loc[split:,features]; y_test=df.loc[split:,"event"]
model=Pipeline([("imputer",SimpleImputer(strategy="median")),
 ("scaler",StandardScaler()),
 ("rf",RandomForestClassifier(n_estimators=350,min_samples_leaf=4,
  class_weight="balanced",random_state=SEED,n_jobs=-1))])
model.fit(X_train,y_train)
prob=model.predict_proba(X_test)[:,1]; pred=(prob>=.5).astype(int)
print(classification_report(y_test,pred,zero_division=0))
print("ROC-AUC:",round(roc_auc_score(y_test,prob),3))
print("PR-AUC:",round(average_precision_score(y_test,prob),3))

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
ConfusionMatrixDisplay.from_predictions(y_test,pred,ax=axs[0],cmap="Blues")
RocCurveDisplay.from_predictions(y_test,prob,ax=axs[1])
axs[0].set_title("Temporal holdout confusion matrix"); axs[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

## 10. Probability calibration
A model can rank observations well while its probabilities are poorly calibrated. The reliability curve compares predicted probability with observed event frequency.

In [ ]:
frac_pos,mean_pred=calibration_curve(y_test,prob,n_bins=10,strategy="quantile")
plt.figure(figsize=(6,6))
plt.plot(mean_pred,frac_pos,marker="o",label="Random Forest")
plt.plot([0,1],[0,1],"--",label="Perfect calibration")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed event fraction")
plt.title("Calibration curve"); plt.legend(); plt.show()

## 11. Model disagreement as an uncertainty proxy
Variation among tree predictions provides a rough disagreement signal. It is not a formal confidence interval or guarantee of coverage.

In [ ]:
Xt=model.named_steps["scaler"].transform(model.named_steps["imputer"].transform(X_test))
tree_probs=np.vstack([tree.predict_proba(Xt)[:,1] for tree in model.named_steps["rf"].estimators_])
tree_mean=tree_probs.mean(axis=0); tree_std=tree_probs.std(axis=0)
uncertainty=pd.DataFrame({"time_index":df.loc[split:,"time_index"].to_numpy(),
 "mean_probability":tree_mean,"tree_disagreement":tree_std,"actual_event":y_test.to_numpy()})
display(uncertainty.nlargest(10,"tree_disagreement"))
plt.figure(figsize=(13,4))
plt.plot(uncertainty.time_index,tree_mean,label="mean prediction")
plt.fill_between(uncertainty.time_index,np.clip(tree_mean-2*tree_std,0,1),
 np.clip(tree_mean+2*tree_std,0,1),alpha=.2,label="± 2 tree SD")
plt.title("Prediction and model-disagreement band"); plt.legend(); plt.show()

## 12. Baseline-relative change score
A rolling standardized deviation from the baseline highlights sustained changes. This is an exploratory score, not a formal change-point test.

In [ ]:
base=df.loc[:baseline_end-1,["load","latency","vibration"]]
z=(df[["load","latency","vibration"]]-base.mean())/base.std().replace(0,1)
change_score=z.rolling(160,min_periods=80).mean().abs().mean(axis=1)
plt.figure(figsize=(14,4)); plt.plot(t,change_score)
plt.axhline(1,ls="--",label="illustrative threshold")
plt.title("Baseline-relative change score"); plt.xlabel("Time"); plt.legend(); plt.show()

## 13. Explainability with permutation importance
Feature importance measures predictive usefulness on the held-out period. It does not prove causal influence.

In [ ]:
perm=permutation_importance(model,X_test,y_test,scoring="roc_auc",n_repeats=8,random_state=SEED,n_jobs=-1)
importance=pd.DataFrame({"feature":features,"importance":perm.importances_mean,
 "std":perm.importances_std}).sort_values("importance")
display(importance.sort_values("importance",ascending=False).round(4))
plt.figure(figsize=(9,5)); plt.barh(importance.feature,importance.importance,xerr=importance["std"])
plt.title("Permutation importance"); plt.xlabel("Decrease in ROC-AUC after shuffling")
plt.tight_layout(); plt.show()

## 14. Weak-signal discovery priority
We combine anomaly score, baseline-relative change, and model disagreement into a heuristic ranking. It is not a probability or validated risk score.

In [ ]:
def minmax(s):
    lo,hi=s.min(),s.max()
    return (s-lo)/(hi-lo) if hi>lo else s*0
weak=pd.DataFrame({"time_index":df.loc[split:,"time_index"].to_numpy(),
 "anomaly":minmax(df.loc[split:,"anomaly_score"]).to_numpy(),
 "change":minmax(change_score.loc[split:]).to_numpy(),
 "uncertainty":minmax(uncertainty.tree_disagreement).to_numpy(),
 "predicted_event_probability":prob})
weak["discovery_priority"]=.40*weak.anomaly+.35*weak.change+.25*weak.uncertainty
display(weak.nlargest(15,"discovery_priority"))
plt.figure(figsize=(13,4)); plt.plot(weak.time_index,weak.discovery_priority,lw=.8)
plt.title("Composite weak-signal discovery priority"); plt.xlabel("Time"); plt.ylabel("Relative score")
plt.show()

## 15. Investigate the highest-priority observations

In [ ]:
top_times=weak.nlargest(12,"discovery_priority").time_index.astype(int).tolist()
display(df.loc[top_times,["time_index"]+features+["event","injected_anomaly","anomaly_score","anomaly_flag"]]
 .sort_values("anomaly_score",ascending=False).round(3))

## 16. Executive intelligence dashboard

In [ ]:
fig,axs=plt.subplots(2,2,figsize=(14,9))
axs[0,0].hist(prob,bins=25); axs[0,0].set_title("Predicted event probabilities")
axs[0,1].plot(t,df.anomaly_score,lw=.6); axs[0,1].set_title("Anomaly score over time")
axs[1,0].plot(t,change_score,lw=.7); axs[1,0].set_title("Baseline-relative change")
axs[1,1].plot(weak.time_index,weak.discovery_priority,lw=.7); axs[1,1].set_title("Weak-signal priority")
plt.tight_layout(); plt.show()

## 17. Automated findings summary

In [ ]:
print("="*60)
print("SIGNAL//SILENCE — EXPERIMENT SUMMARY")
print("="*60)
print("Synthetic records:",len(df))
print("Injected point anomalies:",int(df.injected_anomaly.sum()))
print("Anomaly flags:",int(df.anomaly_flag.sum()))
print("Temporal test observations:",len(y_test))
print("Event ROC-AUC:",round(roc_auc_score(y_test,prob),3))
print("Event PR-AUC:",round(average_precision_score(y_test,prob),3))
print("Event F1 at threshold 0.50:",round(f1_score(y_test,pred),3))
print("Top predictive feature:",importance.iloc[-1].feature)
print("Top investigation timestamps:",top_times[:5])
print("\nThese are synthetic benchmark results, not real-world findings.")

## 18. Export reusable outputs

In [ ]:
df.to_csv("signal_silence_synthetic_data.csv",index=False)
uncertainty.to_csv("signal_silence_prediction_uncertainty.csv",index=False)
weak.to_csv("signal_silence_weak_signal_rankings.csv",index=False)
importance.to_csv("signal_silence_feature_importance.csv",index=False)
print("Exported dataset, uncertainty, weak-signal rankings, and feature importance CSV files.")

## 19. Limitations and next research steps

- Synthetic data and injected anomalies simplify real operating environments.
- Isolation Forest may flag legitimate drift as unusual.
- Tree disagreement is a rough uncertainty proxy, not a calibrated interval.
- The composite priority score is heuristic and needs validation.
- A single temporal split does not establish robust generalization.

**Next steps:** validate on a documented real dataset; compare multiple anomaly detectors; add formal change-point detection; assess calibration and subgroup performance; test across multiple independent time windows; and introduce human review.

### Final thought
**The goal is not to make a model sound certain. It is to build analytical systems that detect subtle change, expose uncertainty, and direct attention toward signals worth understanding.**